# 09 — Hyperparameter Tuning (Hyperopt)
## Customer Analytics Platform

Purpose: Tune the V2 CLV model (recency, frequency, monetary only)
using Hyperopt's Bayesian search (TPE), track every trial in MLflow,
and register the best-performing config as V3.

Note: this notebook tunes the model, it does not fix the target leakage
noted in 06_CLV_Model. Metrics here are still not reliable for reporting
CLV performance — see 06 for the temporal-split fix needed for that.

In [0]:
# install hyperopt alongside existing ML libraries

%pip install hyperopt xgboost mlflow scikit-learn

In [0]:
# restart kernel to use updated packages

dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import pandas as pd
import numpy as np

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

print("hyperopt and ml libraries ready")

In [0]:
# load gold layer rfm features - same source as 06_CLV_Model

gold_path = "/Volumes/workspace/default/olist_raw_data/gold"

gold_df = spark.read.format("delta") \
    .load(f"{gold_path}/rfm_features")

pdf = gold_df.select(
    "recency",
    "frequency",
    "monetary",
    "is_high_value"
).toPandas()

pdf = pdf.fillna(0)

print(f"dataset shape : {pdf.shape}")
print(pdf["is_high_value"].value_counts())

In [0]:
# keep only recency, frequency, monetary - same feature set as v2
# tuning stays on the leakage-reduced feature set, not the full leaky one

X = pdf[["recency", "frequency", "monetary"]]
y = pdf["is_high_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"train : {X_train.shape[0]} rows")
print(f"test  : {X_test.shape[0]} rows")

In [0]:
# define the hyperopt search space
# ranges kept realistic for a 93k row, 3 feature dataset
# no point searching huge depth/estimators, will just overfit faster

space = {
    "max_depth":        hp.quniform("max_depth", 3, 8, 1),
    "learning_rate":    hp.loguniform("learning_rate", np.log(0.01), np.log(0.3)),
    "n_estimators":     hp.quniform("n_estimators", 50, 300, 10),
    "subsample":        hp.uniform("subsample", 0.6, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
    "min_child_weight": hp.quniform("min_child_weight", 1, 10, 1),
    "gamma":            hp.uniform("gamma", 0, 5)
}

In [0]:
# objective function
# each trial trains on train split, scores with 5-fold cv roc_auc
# logs every trial to mlflow as a nested run so all 30+ attempts are tracked

mlflow.set_experiment(
    "/Users/elitahazelgorimanikonda@gmail.com/CLV_Customer_Segmentation"
)

def objective(params):
    params["max_depth"]        = int(params["max_depth"])
    params["n_estimators"]     = int(params["n_estimators"])
    params["min_child_weight"] = int(params["min_child_weight"])

    with mlflow.start_run(nested=True, run_name="hyperopt_trial"):
        model = xgb.XGBClassifier(
            **params,
            random_state=42,
            eval_metric="logloss",
            verbosity=0
        )

        cv_scores = cross_val_score(
            model, X_train, y_train,
            cv=5, scoring="roc_auc", n_jobs=-1
        )
        mean_auc = cv_scores.mean()

        mlflow.log_params(params)
        mlflow.log_metric("cv_roc_auc_mean", mean_auc)
        mlflow.log_metric("cv_roc_auc_std", cv_scores.std())

    # hyperopt minimizes, so return negative auc
    return {"loss": -mean_auc, "status": STATUS_OK}

In [0]:
# run the search
# 40 evals is a reasonable budget for 3 features - diminishing returns past that

trials = Trials()

with mlflow.start_run(run_name="hyperopt_search_v3"):
    best_params = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=40,
        trials=trials
    )

    mlflow.log_param("n_trials", 40)
    mlflow.log_metric("best_cv_roc_auc", -min(trials.losses()))

print("search complete")
print("\nbest raw params (need int casting below):")
print(best_params)

In [0]:
# cast params properly and train the final v3 model on full training set

best_params_final = {
    "max_depth":        int(best_params["max_depth"]),
    "learning_rate":    best_params["learning_rate"],
    "n_estimators":     int(best_params["n_estimators"]),
    "subsample":        best_params["subsample"],
    "colsample_bytree": best_params["colsample_bytree"],
    "min_child_weight": int(best_params["min_child_weight"]),
    "gamma":            best_params["gamma"]
}

print("final tuned params:")
for k, v in best_params_final.items():
    print(f"  {k:<18}: {v}")

In [0]:
# train and log final tuned model as v3, compare against v2 baseline

with mlflow.start_run(run_name="xgboost_clv_v3_tuned"):

    model_v3 = xgb.XGBClassifier(
        **best_params_final,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    model_v3.fit(X_train, y_train)

    y_pred      = model_v3.predict(X_test)
    y_pred_prob = model_v3.predict_proba(X_test)[:, 1]

    accuracy  = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall    = recall_score(y_test, y_pred)
    f1        = f1_score(y_test, y_pred)
    roc_auc   = roc_auc_score(y_test, y_pred_prob)

    mlflow.log_params(best_params_final)
    mlflow.log_param("features", "recency, frequency, monetary")
    mlflow.log_param("tuning_method", "hyperopt_tpe_40_evals")

    mlflow.log_metric("accuracy",  accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall",    recall)
    mlflow.log_metric("f1_score",  f1)
    mlflow.log_metric("roc_auc",   roc_auc)

    mlflow.xgboost.log_model(model_v3, "xgboost_clv_v3_tuned")

    print("V3 Tuned Model Complete")
    print("-" * 40)
    print(f"accuracy  : {accuracy:.4f}")
    print(f"precision : {precision:.4f}")
    print(f"recall    : {recall:.4f}")
    print(f"f1 score  : {f1:.4f}")
    print(f"roc auc   : {roc_auc:.4f}")
    print("-" * 40)
    print("v3 logged to mlflow")

In [0]:
# feature importance for the tuned model - sanity check against v2

import matplotlib.pyplot as plt

feature_names = ["recency", "frequency", "monetary"]
importance    = model_v3.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_names, importance)
plt.xlabel("importance score")
plt.title("XGBoost Feature Importance — V3 Tuned")
plt.tight_layout()
plt.show()

for name, score in zip(feature_names, importance):
    print(f"  {name:<12} : {score:.4f}")

## Hyperparameter Tuning Summary

Method: Hyperopt Tree-structured Parzen Estimator (TPE), 40 trials,
5-fold CV on ROC-AUC, all trials logged to MLflow as nested runs.

Model: XGBoost Customer Value Classifier V3
Features: recency, frequency, monetary (same as V2, no score columns)

Important: an improvement here over V2 is expected and does not mean
the leakage problem is solved — recency/frequency/monetary are still
the exact values the target was derived from. This tuning exercise is
valid to keep as a portfolio artifact (shows Hyperopt/MLflow competence),
but should not be presented as a performance win on its own.

Next Step:
- Apply same search space once the temporal-split target exists
- That result is the one worth quoting for actual CLV performance